# Prepare C-MAPSS for AutoML regression
Read the registered AML folder asset **nasa-cmapss:1** in the AML Python SDK v2 kernel. No local files from the download notebook, package installations, or repository setup are needed. The connection retains your working Azure Government settings.

The notebook resolves the asset's workspace Blob datastore and reads the train, test and RUL files through the Blob SDK already used by the AML SDK. It uses the datastore's configured credentials, or your current identity for identity-based storage. Credentials are not printed or written to output. It does not import MLTable, azureml-fsspec, or azureml.dataprep.

Start with **FD001** (one operating condition; HPC degradation). Validate observations, calculate **rul_cycles** without a cap, and export AutoML tables. Training and validation use different engines. NASA test data stays separate for final evaluation. The raw asset is never changed; this notebook does not submit training.

Source: A. Saxena and K. Goebel (2008), *Turbofan Engine Degradation Simulation Data Set*, NASA Ames Research Center. Simulation results are not certified maintenance advice.

## Sensor meanings
Names follow **Table 2, page 3, of Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation**, in the dataset's sensor order. These are documented channel definitions, not guesses based only on correlation.

The measurements distinguish fan inlet, low-pressure compressor (LPC) outlet, high-pressure compressor (HPC) outlet, and low-pressure turbine (LPT) outlet. They also distinguish total/static pressure, physical/corrected shaft speed, and high-/low-pressure turbine (HPT/LPT) coolant bleed. They do not identify individual blade rows within those components.

Temperatures remain in degrees Rankine, pressures in psia, speeds in rpm where unambiguous, and coolant flows in lbm/s. No unit conversions are applied. The demanded corrected fan-speed channel is 100 in FD001 although the paper lists rpm, so its name deliberately has no unit suffix. Bleed enthalpy likewise has no assumed physical unit. The paper's HPT-outlet temperature T48 and stall margins were not provided as participant sensor data and are not invented here.

The three operating settings remain numbered: their ranges suggest altitude, Mach, and throttle respectively, but the readme does not define their individual units. Its last line also contains a numbering typo: 26 total columns means 21 sensor columns, not 26 sensors. The original sensor-number-to-name mapping is saved with every prepared asset.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.parse import urlparse, unquote
import json
import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import AzureAuthorityHosts, DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

ASSET_NAME = 'nasa-cmapss'
ASSET_VERSION = '1'
SUBSET = 'FD001'
SEED = 42
OUTPUT_DIR = Path('data/prepared') / SUBSET

credential = DefaultAzureCredential(authority=AzureAuthorityHosts.AZURE_GOVERNMENT)
client = MLClient.from_config(credential=credential, cloud='AzureUSGovernment')
raw_asset = client.data.get(name=ASSET_NAME, version=ASSET_VERSION)
asset_uri = urlparse(raw_asset.path)
assert asset_uri.scheme == 'azureml', 'Expected an AML datastore URI for the registered asset'
datastore_name, separator, blob_prefix = asset_uri.path.partition('/datastores/')[2].partition('/paths/')
assert separator, 'Expected a datastore folder path'
blob_prefix = unquote(blob_prefix).strip('/')
datastore = client.datastores.get(datastore_name, include_secrets=True)
assert datastore.type == 'AzureBlob', 'Expected the workspace Blob datastore'
storage_credential = (
    getattr(datastore.credentials, 'account_key', None)
    or getattr(datastore.credentials, 'sas_token', None)
    or credential
)
source_tables = {}
with BlobServiceClient(
    account_url=f'https://{datastore.account_name}.blob.{datastore.endpoint}',
    credential=storage_credential,
) as storage:
    for kind in ('train', 'test', 'RUL'):
        blob_name = f'{blob_prefix}/{kind}_{SUBSET}.txt'.lstrip('/')
        content = storage.get_blob_client(datastore.container_name, blob_name).download_blob().readall()
        source_tables[kind] = pd.read_csv(BytesIO(content), sep=r"\s+", header=None)

sensor_column_mapping = {
    'sensor_1': 't2_fan_inlet_total_temperature_rankine',
    'sensor_2': 't24_lpc_outlet_total_temperature_rankine',
    'sensor_3': 't30_hpc_outlet_total_temperature_rankine',
    'sensor_4': 't50_lpt_outlet_total_temperature_rankine',
    'sensor_5': 'p2_fan_inlet_pressure_psia',
    'sensor_6': 'p15_bypass_duct_total_pressure_psia',
    'sensor_7': 'p30_hpc_outlet_total_pressure_psia',
    'sensor_8': 'nf_physical_fan_speed_rpm',
    'sensor_9': 'nc_physical_core_speed_rpm',
    'sensor_10': 'epr_engine_pressure_ratio',
    'sensor_11': 'ps30_hpc_outlet_static_pressure_psia',
    'sensor_12': 'phi_fuel_flow_to_hpc_static_pressure_ratio',
    'sensor_13': 'nrf_corrected_fan_speed_rpm',
    'sensor_14': 'nrc_corrected_core_speed_rpm',
    'sensor_15': 'bpr_bypass_ratio',
    'sensor_16': 'farb_burner_fuel_air_ratio',
    'sensor_17': 'htbleed_bleed_enthalpy',
    'sensor_18': 'nf_dmd_demanded_fan_speed_rpm',
    'sensor_19': 'pcnfr_dmd_demanded_corrected_fan_speed',
    'sensor_20': 'w31_hpt_coolant_bleed_lbm_per_s',
    'sensor_21': 'w32_lpt_coolant_bleed_lbm_per_s',
}
columns = ['engine_id', 'cycle', 'setting_1', 'setting_2', 'setting_3'] + list(sensor_column_mapping.values())
print(f'Source asset: {raw_asset.id}')

In [ ]:
def read_trajectories(frame):
    frame = frame.copy()
    assert frame.shape[1] == 26, 'Expected 26 columns'
    frame.columns = columns
    frame = frame.apply(pd.to_numeric, errors='raise')
    assert not frame.isna().any().any(), 'Missing values'
    assert not frame.isin([float('inf'), float('-inf')]).any().any(), 'Infinite values'
    assert not frame.duplicated(['engine_id', 'cycle']).any(), 'Duplicate engine/cycle'
    for name in ('engine_id', 'cycle'):
        assert frame[name].gt(0).all() and frame[name].mod(1).eq(0).all(), f'Invalid {name}'
        frame[name] = frame[name].astype(int)
    assert frame.groupby('engine_id')['cycle'].first().eq(1).all(), 'Engines must start at cycle 1'
    assert frame.groupby('engine_id')['cycle'].diff().dropna().eq(1).all(), 'Unordered or missing cycles'
    return frame.sort_values(['engine_id', 'cycle']).reset_index(drop=True)

training = read_trajectories(source_tables['train'])
test_trajectories = read_trajectories(source_tables['test'])
truth = source_tables['RUL']
assert truth.shape == (test_trajectories['engine_id'].nunique(), 1), 'RUL count does not match test engines'
truth = pd.to_numeric(truth.iloc[:, 0], errors='raise')
assert truth.notna().all() and truth.ge(0).all() and truth.mod(1).eq(0).all(), 'Invalid RUL labels'
assert sorted(test_trajectories['engine_id'].unique()) == list(range(1, len(truth) + 1)), 'RUL rows require consecutive engine IDs starting at 1'
print(f'{SUBSET}: {len(training):,} training rows, {training.engine_id.nunique()} training engines; no missing or duplicate observations.')

## Label and split
For run-to-failure training engines, RUL is the engine's final cycle minus the current cycle. Reserve 20% of engines for validation. Only training engines determine which constant inputs to remove; scaling and imputation are left to AutoML.

For the final test, keep one last observation per engine and attach its supplied RUL. Test engine IDs belong to a separate fleet, even where their numbers overlap training IDs. Engine IDs and future-derived values are excluded from model inputs.

In [ ]:
training['rul_cycles'] = training.groupby('engine_id')['cycle'].transform('max') - training['cycle']
engines = pd.Series(sorted(training['engine_id'].unique())).sample(frac=1, random_state=SEED).tolist()
assert len(engines) >= 2, 'At least two training engines are required'
validation_count = max(1, round(len(engines) * 0.2))
validation_engines = engines[:validation_count]
train_engines = engines[validation_count:]
assert set(train_engines).isdisjoint(validation_engines)

train_rows = training[training['engine_id'].isin(train_engines)].copy()
validation_rows = training[training['engine_id'].isin(validation_engines)].copy()
candidate_features = columns[2:]
constant_features = [name for name in candidate_features if train_rows[name].nunique() == 1]
features = ['cycle'] + [name for name in candidate_features if name not in constant_features]

test_rows = test_trajectories.groupby('engine_id', sort=True).tail(1).copy()
rul_by_engine = pd.Series(truth.to_numpy(), index=range(1, len(truth) + 1))
test_rows['rul_cycles'] = test_rows['engine_id'].map(rul_by_engine).astype(int)
tables = {
    'train': train_rows[features + ['rul_cycles']],
    'validation': validation_rows[features + ['rul_cycles']],
    'test': test_rows[features + ['rul_cycles']],
}

full_training = training.copy()
full_training['source_split'] = full_training['engine_id'].map(
    lambda engine: 'train' if engine in train_engines else 'validation'
)
full_training['engine_uid'] = SUBSET + '_train_' + full_training['engine_id'].astype(str)
full_test = test_trajectories.copy()
full_test['rul_cycles'] = (
    full_test.groupby('engine_id')['cycle'].transform('max')
    - full_test['cycle']
    + full_test['engine_id'].map(rul_by_engine)
).astype(int)
full_test['source_split'] = 'test'
full_test['engine_uid'] = SUBSET + '_test_' + full_test['engine_id'].astype(str)
tables['full'] = pd.concat([full_training, full_test], ignore_index=True)
assert not tables['full'].duplicated(['engine_uid', 'cycle']).any()
print('Removed training-constant inputs from split tables only:', constant_features)
for split, table in tables.items():
    assert not table.isna().any().any()
    print(f'{split}: {len(table):,} rows, {len(table.columns)} columns')
tables['full'].head()

In [ ]:
for split, table in tables.items():
    folder = OUTPUT_DIR / split
    folder.mkdir(parents=True, exist_ok=True)
    table.to_csv(folder / 'data.csv', index=False)
    (folder / 'MLTable').write_text(
        "paths:\n  - file: ./data.csv\ntransformations:\n  - read_delimited:\n      delimiter: ','\n      encoding: utf8\n",
        encoding='utf-8',
    )

preparation = {
    'source_asset_id': raw_asset.id,
    'source_files': [f'{kind}_{SUBSET}.txt' for kind in ('train', 'test', 'RUL')],
    'subset': SUBSET, 'seed': SEED, 'target': 'rul_cycles', 'rul_cap': None,
    'citation': 'A. Saxena and K. Goebel (2008), Turbofan Engine Degradation Simulation Data Set, NASA Ames Research Center.',
    'sensor_definition_source': 'Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation, Table 2, page 3',
    'sensor_column_mapping': sensor_column_mapping,
    'sensor_unit_notes': 'No unit conversions. PCNfR_dmd scale is ambiguous; htBleed physical unit is unspecified. Operating settings remain numbered.',
    'train_engines': sorted(train_engines), 'validation_engines': sorted(validation_engines),
    'test_engines_in_row_order': test_rows['engine_id'].tolist(),
    'features': features, 'removed_constant_features': constant_features,
}
(OUTPUT_DIR / 'preparation.json').write_text(json.dumps(preparation, indent=2) + '\n', encoding='utf-8')
print(f'Prepared tables: {OUTPUT_DIR.resolve()}')

## Register prepared tables for AutoML
Run the next cell to upload the prepared files to workspace storage and register four **MLTable data assets** in AML: **cmapss-fd001-train**, **cmapss-fd001-validation**, **cmapss-fd001-test**, and **cmapss-fd001-full**. Names follow the selected SUBSET. The local output folders are staging only; the registered assets persist independently of this notebook's files. Each uploaded folder includes its CSV, MLTable definition, and preparation record. Storage charges may apply.

Use a new **PREPARED_VERSION** when changing an already registered preparation. The cell prints each registered asset ID and storage path. No training is launched.

### Full dataset for your own splits
The **full** asset (also staged at **data/prepared/FD001/full/data.csv**) contains all observed cycles from the selected subset's original train and test files, including every training and validation row. Unlike the ready-made test asset, it keeps all test observations, not just the last cycle. It retains all sensor and setting columns, including constants; no missing future observations are fabricated.

**rul_cycles** uses final training cycle minus current cycle for run-to-failure engines. For truncated test engines it uses last observed cycle minus current cycle plus NASA's supplied RUL at the last observation. **engine_id** is the original NASA number. **engine_uid**, such as FD001_train_1 or FD001_test_1, uniquely identifies an engine across the two fleets. **source_split** records the existing train, validation, or test membership.

Make your own splits by **engine_uid**, not by rows or engine_id alone. Exclude **engine_uid**, **engine_id**, and **source_split** from model inputs, and use **rul_cycles** only as the target. Fit preprocessing and constant-column removal on your chosen training engines only. If you use NASA test engines for training or tuning, they are no longer an untouched official test benchmark; reserve a new engine-disjoint holdout before fitting.

### Existing ready-made splits
For AutoML, choose **Regression**, target **rul_cycles**, and primary metric **normalized_root_mean_squared_error**. Supply the registered **train** asset as training data and **validation** as explicit validation data. Do not use random row splitting or automatic cross-validation, which can mix the same engine across splits. Keep **test** for final evaluation only.

In [ ]:
from azure.ai.ml.entities import Data

PREPARED_VERSION = '1'
registered_assets = {}
for split in tables:
    folder = OUTPUT_DIR / split
    (folder / 'preparation.json').write_text(
        (OUTPUT_DIR / 'preparation.json').read_text(encoding='utf-8'),
        encoding='utf-8',
    )
    registered_assets[split] = client.data.create_or_update(Data(
        name=f'cmapss-{SUBSET.lower()}-{split}',
        version=PREPARED_VERSION,
        type='mltable',
        path=str(folder),
        description=f'{SUBSET} {split} data for RUL regression; target: rul_cycles',
        tags={'source_asset_id': raw_asset.id, 'subset': SUBSET},
    ))
    print(f'{split}: {registered_assets[split].id}')
    print(f'Storage: {registered_assets[split].path}')